# 层次聚类可视化

## 目标

理解层次聚类算法的工作原理和可视化。

- 理解层次聚类的迭代过程
- 学习不同的链接方式
- 绘制和解读树状图
- 选择最佳聚类数的方法
- 与 K-Means 对比

## 1. 数据准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.preprocessing import StandardScaler
import seaborn as sns

np.random.seed(42)
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 生成合成数据
def generate_hierarchical_datasets():
    datasets = {}
    
    # 标准球形簇
    X_blobs, y_blobs = make_blobs(
        n_samples=150, n_features=2, centers=4,
        cluster_std=1.5, random_state=42
    )
    datasets['blobs'] = (X_blobs, y_blobs, '标准球形簇')
    
    # 链式数据（适合单链接）
    X_chain = []
    for i in range(3):
        cluster_center = [i * 8, 0]
        chain_cluster = np.random.randn(50, 2) * 0.5 + cluster_center
        # 添加链式结构
        for j in range(5):
            chain_cluster[j*10:(j+1)*10, 0] += j * 1.5
        X_chain.append(chain_cluster)
    X_chain = np.vstack(X_chain)
    y_chain = np.repeat([0, 1, 2], 50)
    datasets['chain'] = (X_chain, y_chain, '链式数据')
    
    # 不同大小和密度的簇
    X_varied, y_varied = make_blobs(
        n_samples=150, n_features=2, centers=3,
        cluster_std=[0.5, 2.0, 1.5], random_state=42
    )
    datasets['varied'] = (X_varied, y_varied, '不同密度簇')
    
    # 环形数据
    X_circles, y_circles = make_circles(
        n_samples=150, factor=0.5, noise=0.05, random_state=42
    )
    datasets['circles'] = (X_circles, y_circles, '环形数据')
    
    return datasets

datasets = generate_hierarchical_datasets()

print("生成的数据集:")
for name, (X, y, desc) in datasets.items():
    print(f"  {name}: {X.shape} - {desc}")

In [ ]:
# 可视化数据集
plt.figure(figsize=(12, 3))

for i, (name, (X, y, desc)) in enumerate(datasets.items()):
    plt.subplot(1, 4, i+1)
    plt.scatter(X[:, 0], X[:, 1], c=y if y is not None else 'gray', 
               cmap='viridis', alpha=0.6, s=30)
    plt.xlabel('特征 1')
    plt.ylabel('特征 2')
    plt.title(f'{desc}\n({name})')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. 层次聚类基础

In [ ]:
# 使用标准球形数据进行层次聚类
X_blobs, y_blobs, _ = datasets['blobs']

# 层次聚类
agg = AgglomerativeClustering(n_clusters=4, linkage='ward')
labels_agg = agg.fit_predict(X_blobs)

print("层次聚类结果:")
print(f"  簇数量: {len(np.unique(labels_agg))}")
print(f"  簇大小分布: {np.bincount(labels_agg)}")
print(f"  链接方式: ward")

In [ ]:
# 可视化聚类结果
plt.figure(figsize=(10, 6))

# 原始数据
plt.subplot(1, 2, 1)
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', alpha=0.6, s=30)
plt.xlabel('特征 1')
plt.ylabel('特征 2')
plt.title('原始标签')
plt.grid(True, alpha=0.3)

# 聚类结果
plt.subplot(1, 2, 2)
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels_agg, cmap='viridis', alpha=0.6, s=30)
plt.xlabel('特征 1')
plt.ylabel('特征 2')
plt.title('层次聚类结果 (Ward)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 计算评估指标
ari = adjusted_rand_score(y_blobs, labels_agg)
silhouette = silhouette_score(X_blobs, labels_agg)

print(f"\n评估指标:")
print(f"  ARI (调整兰德指数): {ari:.3f}")
print(f"  轮廓系数: {silhouette:.3f}")

## 3. 树状图（Dendrogram）

In [ ]:
# 绘制树状图
def plot_dendrogram(X, title, linkage_method='ward', truncate_mode=None, p=30):
    """绘制树状图"""
    # 计算链接矩阵
    Z = linkage(X, method=linkage_method)
    
    plt.figure(figsize=(12, 6))
    dendrogram(
        Z,
        truncate_mode=truncate_mode,
        p=p,
        leaf_rotation=90,
        leaf_font_size=10,
        show_contracted=True
    )
    plt.title(f'{title} ({linkage_method} 链接)')
    plt.xlabel('样本索引或簇大小')
    plt.ylabel('距离')
    plt.axhline(y=10, color='red', linestyle='--', alpha=0.5, label='切割线')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    return Z

# 绘制完整树状图
Z_blobs = plot_dendrogram(X_blobs, '层次聚类树状图', linkage_method='ward')

In [ ]:
# 绘制截断的树状图（更清晰）
Z_blobs_truncated = plot_dendrogram(
    X_blobs, 
    '层次聚类树状图 (截断)', 
    linkage_method='ward', 
    truncate_mode='lastp', 
    p=10
)

## 4. 不同链接方式对比

In [ ]:
# 对比不同链接方式
linkage_methods = ['ward', 'complete', 'average', 'single']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, method in enumerate(linkage_methods):
    # 聚类
    agg_temp = AgglomerativeClustering(n_clusters=4, linkage=method)
    labels_temp = agg_temp.fit_predict(X_blobs)
    
    # 计算链接矩阵
    Z_temp = linkage(X_blobs, method=method)
    
    # 绘制聚类结果
    axes[0, i].scatter(X_blobs[:, 0], X_blobs[:, 1], 
                      c=labels_temp, cmap='viridis', alpha=0.6, s=30)
    axes[0, i].set_xlabel('特征 1')
    axes[0, i].set_ylabel('特征 2')
    axes[0, i].set_title(f'{method} 链接')
    axes[0, i].grid(True, alpha=0.3)
    
    # 计算评估指标
    ari = adjusted_rand_score(y_blobs, labels_temp)
    silhouette = silhouette_score(X_blobs, labels_temp)
    
    # 在标题中显示指标
    axes[0, i].set_xlabel(f'ARI: {ari:.3f}\nSilhouette: {silhouette:.3f}')
    
    # 绘制树状图
    dendrogram(
        Z_temp,
        truncate_mode='lastp',
        p=10,
        ax=axes[1, i],
        leaf_rotation=90,
        leaf_font_size=8
    )
    axes[1, i].set_title(f'{method} 树状图')
    axes[1, i].set_xlabel('样本')
    axes[1, i].set_ylabel('距离')

plt.tight_layout()
plt.show()

print("链接方式说明:")
print("- ward: 最小化簇内方差（默认，通常效果最好）")
print("- complete: 完全链接（簇间最大距离）")
print("- average: 平均链接（簇间平均距离）")
print("- single: 单链接（簇间最小距离，可能产生链式效应）")

## 5. 在不同数据集上的表现

In [ ]:
# 在所有数据集上测试层次聚类（使用 ward 链接）
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, (name, (X, y, desc)) in enumerate(datasets.items()):
    # 聚类
    n_clusters = len(np.unique(y)) if y is not None else 3
    agg_temp = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    labels_temp = agg_temp.fit_predict(X)
    
    # 计算评估指标
    ari = adjusted_rand_score(y, labels_temp) if y is not None else np.nan
    silhouette = silhouette_score(X, labels_temp) if len(np.unique(labels_temp)) > 1 else np.nan
    
    # 绘制原始数据
    axes[0, i].scatter(X[:, 0], X[:, 1], c=y if y is not None else 'gray', 
                      cmap='viridis', alpha=0.6, s=30)
    axes[0, i].set_xlabel('特征 1')
    axes[0, i].set_ylabel('特征 2')
    axes[0, i].set_title(f'{desc}\n原始')
    axes[0, i].grid(True, alpha=0.3)
    
    # 绘制聚类结果
    axes[1, i].scatter(X[:, 0], X[:, 1], c=labels_temp, cmap='viridis', alpha=0.6, s=30)
    axes[1, i].set_xlabel('特征 1')
    axes[1, i].set_ylabel('特征 2')
    axes[1, i].set_title(f'层次聚类\nARI: {ari:.2f}')
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 选择最佳聚类数

In [ ]:
# 使用不同的聚类数测试
K_range = range(2, 8)

results = {'K': list(K_range),
          'Silhouette': [],
          'ARI': []}

for K in K_range:
    agg_temp = AgglomerativeClustering(n_clusters=K, linkage='ward')
    labels_temp = agg_temp.fit_predict(X_blobs)
    
    results['Silhouette'].append(silhouette_score(X_blobs, labels_temp))
    results['ARI'].append(adjusted_rand_score(y_blobs, labels_temp))

import pandas as pd
results_df = pd.DataFrame(results)
print("不同聚类数的评估结果:")
print(results_df.round(3).to_string(index=False))

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Silhouette
axes[0].plot(K_range, results['Silhouette'], 'b-o', linewidth=2, markersize=6)
axes[0].axhline(y=0, color='gray', linestyle='--')
axes[0].set_xlabel('聚类数 K')
axes[0].set_ylabel('轮廓系数')
axes[0].set_title('轮廓系数 vs K')
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(K_range)

# ARI
axes[1].plot(K_range, results['ARI'], 'r-o', linewidth=2, markersize=6)
axes[1].axhline(y=0, color='gray', linestyle='--')
axes[1].set_xlabel('聚类数 K')
axes[1].set_ylabel('ARI')
axes[1].set_title('ARI vs K')
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(K_range)

plt.tight_layout()
plt.show()

# 找到最佳 K
best_K_silhouette = K_range[np.argmax(results['Silhouette'])]
best_K_ARI = K_range[np.argmax(results['ARI'])]

print(f"\n最佳聚类数:")
print(f"  根据轮廓系数: K = {best_K_silhouette}")
print(f"  根据 ARI: K = {best_K_ARI}")

## 7. 从树状图选择聚类数

In [ ]:
# 绘制带有切割线的树状图
Z_blobs = linkage(X_blobs, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(
    Z_blobs,
    truncate_mode='lastp',
    p=12,
    leaf_rotation=90,
    leaf_font_size=10,
    show_contracted=True
)

# 添加不同高度的切割线
cut_heights = [8, 12, 20]
colors = ['red', 'green', 'blue']
for height, color in zip(cut_heights, colors):
    clusters = fcluster(Z_blobs, height, criterion='distance')
    n_clusters = len(np.unique(clusters))
    plt.axhline(y=height, color=color, linestyle='--', 
               alpha=0.7, linewidth=2, label=f'高度={height}, 簇数={n_clusters}')

plt.title('树状图：不同切割位置的聚类结果')
plt.xlabel('样本')
plt.ylabel('距离')
plt.legend()
plt.tight_layout()
plt.show()

print("树状图解读:")
print("- 纵轴是合并距离（簇间距离）")
print("- 横轴是样本点")
print("- 切割位置决定了聚类数")
print("- 选择"肘部"位置作为切割点")
print("\n切割原则:")
print("1. 观察树状图中的"台阶"或"肘部"")
print("2. 在合并距离变化最大的位置切割")
print("3. 结合业务需求确定最终簇数")

## 8. 层次聚类迭代过程可视化

In [ ]:
# 可视化层次聚类的逐步合并过程
def visualize_hierarchical_steps(X, n_steps_to_show=6):
    """可视化层次聚类的逐步合并过程"""
    Z = linkage(X, method='ward')
    
    # 选择显示的步骤
    total_steps = len(Z)
    selected_steps = list(range(0, total_steps, max(1, total_steps // n_steps_to_show)))
    if len(selected_steps) > n_steps_to_show:
        selected_steps = selected_steps[:n_steps_to_show]
    
    fig, axes = plt.subplots(1, len(selected_steps), figsize=(15, 3))
    
    for idx, step in enumerate(selected_steps):
        # 从链接矩阵重建聚类标签
        n_samples = X.shape[0]
        clusters = fcluster(Z, step + 1, criterion='maxclust')
        
        # 绘制
        axes[idx].scatter(X[:, 0], X[:, 1], c=clusters, cmap='viridis', alpha=0.6, s=30)
        axes[idx].set_xlabel('特征 1')
        axes[idx].set_ylabel('特征 2')
        axes[idx].set_title(f'步骤 {step}: {len(np.unique(clusters))} 个簇')
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 使用较小的数据集可视化
X_small, y_small, _ = datasets['blobs']
X_small = X_small[:50]  # 只用前50个点
y_small = y_small[:50]

visualize_hierarchical_steps(X_small, n_steps_to_show=5)

## 9. 链式数据上的表现

In [ ]:
# 在链式数据上测试不同链接方式
X_chain, y_chain, _ = datasets['chain']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, method in enumerate(linkage_methods):
    # 聚类
    agg_temp = AgglomerativeClustering(n_clusters=3, linkage=method)
    labels_temp = agg_temp.fit_predict(X_chain)
    
    # 计算评估指标
    ari = adjusted_rand_score(y_chain, labels_temp)
    silhouette = silhouette_score(X_chain, labels_temp)
    
    # 绘制原始数据
    axes[0, i].scatter(X_chain[:, 0], X_chain[:, 1], c=y_chain, cmap='viridis', alpha=0.6, s=30)
    axes[0, i].set_xlabel('特征 1')
    axes[0, i].set_ylabel('特征 2')
    axes[0, i].set_title(f'{method} 链接\n原始')
    axes[0, i].grid(True, alpha=0.3)
    
    # 绘制聚类结果
    axes[1, i].scatter(X_chain[:, 0], X_chain[:, 1], c=labels_temp, cmap='viridis', alpha=0.6, s=30)
    axes[1, i].set_xlabel('特征 1')
    axes[1, i].set_ylabel('特征 2')
    axes[1, i].set_title(f'层次聚类\nARI: {ari:.3f}')
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("链式数据分析:")
print("- 单链接 (single): 适合链式数据，可以正确识别长条形簇")
print("- Ward/complete/average: 可能将链式数据错误地分割成多个球形簇")

## 10. 层次聚类 vs K-Means

In [ ]:
from sklearn.cluster import KMeans

# 对比层次聚类和 K-Means
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, (name, (X, y, desc)) in enumerate(datasets.items()):
    n_clusters = len(np.unique(y)) if y is not None else 3
    
    # 层次聚类
    agg_temp = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    labels_agg = agg_temp.fit_predict(X)
    
    # K-Means
    kmeans_temp = KMeans(n_clusters=n_clusters, random_state=42)
    labels_kmeans = kmeans_temp.fit_predict(X)
    
    # 计算指标
    ari_agg = adjusted_rand_score(y, labels_agg) if y is not None else np.nan
    ari_kmeans = adjusted_rand_score(y, labels_kmeans) if y is not None else np.nan
    
    # 绘制层次聚类
    axes[0, i].scatter(X[:, 0], X[:, 1], c=labels_agg, cmap='viridis', alpha=0.6, s=30)
    axes[0, i].set_xlabel('特征 1')
    axes[0, i].set_ylabel('特征 2')
    axes[0, i].set_title(f'{desc}\n层次聚类\nARI: {ari_agg:.2f}')
    axes[0, i].grid(True, alpha=0.3)
    
    # 绘制 K-Means
    axes[1, i].scatter(X[:, 0], X[:, 1], c=labels_kmeans, cmap='viridis', alpha=0.6, s=30)
    axes[1, i].scatter(kmeans_temp.cluster_centers_[:, 0], kmeans_temp.cluster_centers_[:, 1],
               c='black', marker='x', s=100, linewidths=2)
    axes[1, i].set_xlabel('特征 1')
    axes[1, i].set_ylabel('特征 2')
    axes[1, i].set_title(f'{desc}\nK-Means\nARI: {ari_kmeans:.2f}')
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("层次聚类 vs K-Means:")
print("| 特点 | 层次聚类 | K-Means |")
print("|------|---------|----------|")
print("| 需要指定簇数 | 可选（通过树状图选择） | 必须 |")
print("| 层次结构 | 有 | 无 |")
print("| 计算复杂度 | O(n²) 或 O(n³) | O(nkt) |")
print("| 大数据 | 不适合 | 适合 |")
print("| 初始敏感 | 不敏感 | 敏感 |")
print("| 可视化 | 树状图 | 质心图 |")

## 11. 实际应用示例：客户分群

In [ ]:
# 创建模拟客户数据
def create_customer_data(n_customers=200):
    """创建模拟客户数据"""
    np.random.seed(42)
    
    # 生成不同类型的客户
    customer_types = [
        {'name': '高价值', 'n': 50, 'mean': [8000, 50], 'std': [1000, 10]},
        {'name': '中价值', 'n': 80, 'mean': [3000, 30], 'std': [800, 8]},
        {'name': '低价值', 'n': 70, 'mean': [500, 10], 'std': [200, 3]}
    ]
    
    X = []
    y = []
    
    for i, ct in enumerate(customer_types):
        cluster = np.random.randn(ct['n'], 2) * ct['std'] + ct['mean']
        X.append(cluster)
        y.extend([i] * ct['n'])
    
    X = np.vstack(X)
    y = np.array(y)
    
    return X, y

X_customers, y_customers = create_customer_data()

# 特征标准化
scaler = StandardScaler()
X_customers_scaled = scaler.fit_transform(X_customers)

print("客户数据:")
print(f"  样本数: {len(X_customers)}")
print(f"  特征: [年消费额, 访问频率]")
print(f"  真实客户类型数: {len(np.unique(y_customers))}")

In [ ]:
# 层次聚类客户数据
agg_customers = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_customers = agg_customers.fit_predict(X_customers_scaled)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 原始数据（真实标签）
type_names = ['低价值', '中价值', '高价值']
for i, type_name in enumerate(type_names):
    mask = (y_customers == i)
    axes[0].scatter(X_customers[mask, 0], X_customers[mask, 1], 
                  label=type_name, alpha=0.6)
axes[0].set_xlabel('年消费额')
axes[0].set_ylabel('访问频率')
axes[0].set_title('真实客户类型')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 层次聚类结果
cluster_names = ['簇 0', '簇 1', '簇 2']
colors = ['red', 'blue', 'green']
for i, cluster_name in enumerate(cluster_names):
    mask = (labels_customers == i)
    axes[1].scatter(X_customers[mask, 0], X_customers[mask, 1],
                  c=colors[i], label=cluster_name, alpha=0.6)
axes[1].set_xlabel('年消费额')
axes[1].set_ylabel('访问频率')
axes[1].set_title('层次聚类结果')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 树状图
Z_customers = linkage(X_customers_scaled, method='ward')
dendrogram(
    Z_customers,
    truncate_mode='lastp',
    p=15,
    ax=axes[2],
    leaf_rotation=90,
    leaf_font_size=8
)
axes[2].set_title('客户分群树状图')
axes[2].set_xlabel('样本')
axes[2].set_ylabel('距离')

plt.tight_layout()
plt.show()

# 分析聚类结果
ari_customers = adjusted_rand_score(y_customers, labels_customers)
silhouette_customers = silhouette_score(X_customers_scaled, labels_customers)

print(f"\n聚类效果:")
print(f"  ARI: {ari_customers:.3f}")
print(f"  轮廓系数: {silhouette_customers:.3f}")

# 每个簇的特征分析
print(f"\n各簇特征:")
for i in range(3):
    mask = (labels_customers == i)
    cluster_data = X_customers[mask]
    print(f"  簇 {i}: {np.sum(mask)} 个客户")
    print(f"    平均年消费额: {cluster_data[:, 0].mean():.0f}")
    print(f"    平均访问频率: {cluster_data[:, 1].mean():.1f}")

## 12. 层次聚类的局限性

In [ ]:
# 演示层次聚类的局限性

# 1. 计算复杂度演示
import time

sample_sizes = [100, 200, 400, 800]
times_agg = []
times_kmeans = []

print("计算复杂度对比:")
print("样本数\t层次聚类时间\tK-Means时间")
print("-" * 50)

for n in sample_sizes:
    X_temp = np.random.randn(n, 2)
    
    # 层次聚类
    start = time.time()
    agg_temp = AgglomerativeClustering(n_clusters=3, linkage='ward')
    agg_temp.fit_predict(X_temp)
    time_agg = time.time() - start
    times_agg.append(time_agg)
    
    # K-Means
    start = time.time()
    kmeans_temp = KMeans(n_clusters=3, random_state=42)
    kmeans_temp.fit_predict(X_temp)
    time_kmeans = time.time() - start
    times_kmeans.append(time_kmeans)
    
    print(f"{n}\t{time_agg:.4f}s\t\t{time_kmeans:.4f}s")

# 可视化
plt.figure(figsize=(10, 5))
plt.plot(sample_sizes, times_agg, 'b-o', label='层次聚类', linewidth=2)
plt.plot(sample_sizes, times_kmeans, 'r-o', label='K-Means', linewidth=2)
plt.xlabel('样本数')
plt.ylabel('运行时间 (秒)')
plt.title('计算复杂度对比')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n层次聚类的局限性:")
print("1. 计算复杂度高: O(n²) 或 O(n³),不适合大数据")
print("2. 不可逆: 一旦合并,不能撤销")
print("3. 对异常值敏感: 尤其是完全链接方法")
print("4. 链接方式选择: 不同方法结果可能差异很大")
print("5. 内存消耗大: 需要存储完整的距离矩阵")

## 总结

### 层次聚类算法

1. **自底向上（凝聚式）**
   - 每个点初始为一个簇
   - 逐步合并最近的簇
   - 直到只剩一个簇

2. **树状图**
   - 可视化层次结构
   - 纵轴: 合并距离
   - 横轴: 样本点
   - 切割位置决定聚类数

### 链接方式

| 方法 | 定义 | 特点 |
|------|------|------|
| Ward | 最小化方差增量 | 通常效果最好 |
| Complete | 簇间最大距离 | 紧凑簇 |
| Average | 簇间平均距离 | 折中方案 |
| Single | 簇间最小距离 | 适合链式数据 |

### 选择最佳聚类数

1. **观察树状图**
   - 寻找"肘部"位置
   - 合并距离变化大的地方

2. **评估指标**
   - 轮廓系数（越大越好）
   - ARI（如果有真实标签）

### 层次聚类 vs K-Means

选择层次聚类当：
- 数据量不大
- 需要层次结构
- 不确定簇的数量
- 需要可视化

选择 K-Means 当：
- 数据量大
- 只需要单一聚类结果
- 簇是球形的
- 计算效率重要

### 实践建议

- 默认使用 Ward 链接
- 观察树状图选择聚类数
- 对于链式数据尝试单链接
- 大数据考虑使用 BIRCH 或 MiniBatch K-Means
- 结合业务知识确定最终簇数